# Notebook 07: HistGradientBoosting Model
## ML-Based Liquidity and Trading-Risk Intelligence for Indian Equities

**Phase:** Phase 8 Batch 4 (Stages 8.9–8.11)
**Target:** High-Risk Event (`hre_5d_train`, 5-day forward horizon)
**Evaluation Protocol:** 10-Fold Expanding-Window Walk-Forward Validation with 5-Session Purging

### Objectives
1. Train `HistGradientBoostingClassifier` across 10 expanding-window walk-forward folds.
2. Utilize all 36 approved candidate features (including F51 `relative_volume`).
3. Leverage native histogram binning for high computational speed and low memory footprint (<500 MB).
4. Exploit native NaN routing during tree splitting.
5. Compare out-of-fold performance (PR-AUC, ROC-AUC, Brier score, calibration) against Logistic Regression and Random Forest.


In [1]:
# Imports and Environment Setup
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add repository root to path regardless of working directory
current_dir = Path('.').resolve()
if (current_dir / 'data' / 'processed').exists():
    REPO_ROOT = current_dir
elif (current_dir.parent / 'data' / 'processed').exists():
    REPO_ROOT = current_dir.parent
else:
    REPO_ROOT = current_dir
    for p in [current_dir] + list(current_dir.parents):
        if (p / 'data' / 'processed').exists():
            REPO_ROOT = p
            break

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.modeling.preprocessing import build_hgbt_pipeline, get_model_features
from src.modeling.evaluation import ModelEvaluator

print(f'Repository root: {REPO_ROOT}')
print('Environment initialized successfully.')


Repository root: C:\Users\vighn\Downloads\Indian_Equities_Liquidity_Risk_ML-main\Indian_Equities_Liquidity_Risk_ML-main
Environment initialized successfully.


### 1. HGBT Architecture & 36-Feature Specification

HistGradientBoosting operates directly on raw feature matrices without scaling or imputation.
Missing values are assigned to dedicated histogram bins during construction.


In [2]:
hgbt_features = get_model_features('hist_gradient_boosting')
print(f'HGBT Features ({len(hgbt_features)}):\n')
print(', '.join(hgbt_features))

pipe_hgbt = build_hgbt_pipeline()
print('\nHGBT Pipeline:', pipe_hgbt)


HGBT Features (36):

daily_return, return_5d, return_20d, return_z20, abs_return, price_rel_ma20, vol_5d, vol_20d, vol_ratio_5_20, downside_vol_20d, max_drawdown_20d, rsi_14, vol_of_vol, log_volume, volume_z20, amihud_ratio, amihud_ma20, amihud_z20, log_turnover, turnover_ratio, low_volume_streak, daily_range, daily_range_ma20, range_z20, overnight_gap, overnight_gap_abs, cs_spread_est, cs_spread_ma10, excess_return_1d, rolling_beta_60d, rolling_corr_60d, relative_volume, nifty_vol_20d, nifty_return_20d, vol_beta, vix_close

HGBT Pipeline: Pipeline(steps=[('classifier',
                 HistGradientBoostingClassifier(class_weight='balanced',
                                                l2_regularization=0.1,
                                                learning_rate=0.05,
                                                max_iter=200,
                                                random_state=42))])


### 2. Walk-Forward Validation Across 10 Folds

Load the 10-fold walk-forward validation results for HistGradientBoosting.


In [3]:
metrics_path = REPO_ROOT / 'data' / 'interim' / 'phase8_batch4' / 'supervised_fold_metrics.csv'
fold_metrics = pd.read_csv(metrics_path)

hgbt_metrics = fold_metrics[fold_metrics['model_name'] == 'hist_gradient_boosting']
display_cols = [
    'fold', 'val_start', 'val_end', 'val_rows', 'val_prev',
    'pr_auc', 'roc_auc', 'brier_score', 'f1_50', 'calibrated_threshold', 'f1_calib', 'fit_time_sec'
]
hgbt_metrics[display_cols].sort_values('fold')


,fold,val_start,val_end,val_rows,val_prev,pr_auc,roc_auc,brier_score,f1_50,calibrated_threshold,f1_calib,fit_time_sec
4,0,2023-05-10,2023-08-02,15780,0.186122,0.650597,0.873069,0.113074,0.590103,0.46,0.587078,13.171164
9,1,2023-08-03,2023-10-31,15821,0.187077,0.601890,0.863157,0.140679,0.571429,0.42,0.555453,24.616093
14,2,2023-11-01,2024-01-30,15939,0.195079,0.627140,0.859359,0.119352,0.595831,0.62,0.586433,13.621328
19,3,2024-01-31,2024-04-30,16157,0.178883,0.613290,0.852786,0.143408,0.538205,0.52,0.544809,23.926433
24,4,2024-05-02,2024-07-29,16260,0.170541,0.592373,0.853728,0.111504,0.569595,0.68,0.502045,26.734204
29,5,2024-07-30,2024-10-23,16358,0.193606,0.638130,0.856317,0.117251,0.578750,0.50,0.578750,14.661388
34,6,2024-10-24,2025-01-20,16435,0.177609,0.618625,0.858395,0.118663,0.569030,0.50,0.569030,13.768176
39,7,2025-01-21,2025-04-21,16685,0.179143,0.595775,0.849091,0.167427,0.514038,0.54,0.526635,17.914156
44,8,2025-04-22,2025-07-15,16911,0.177577,0.584358,0.836544,0.111054,0.532460,0.70,0.389746,19.635791
49,9,2025-07-16,2025-10-10,17011,0.184351,0.584362,0.845872,0.123901,0.562491,0.40,0.548628,22.308934


### 3. Comprehensive Model Comparison Leaderboard

Compare all 5 supervised configurations ranked by Mean Out-of-Fold PR-AUC.


In [4]:
comp_path = REPO_ROOT / 'data' / 'interim' / 'phase8_batch4' / 'supervised_model_comparison.csv'
comp_df = pd.read_csv(comp_path)

leaderboard_cols = [
    'model_name', 'mean_pr_auc', 'std_pr_auc', 'min_pr_auc', 'pooled_pr_auc',
    'mean_roc_auc', 'mean_brier_score', 'mean_f1_50', 'mean_f1_calib',
    'mean_calib_threshold', 'total_fit_time_sec'
]
comp_df[leaderboard_cols]


,model_name,mean_pr_auc,std_pr_auc,min_pr_auc,pooled_pr_auc,mean_roc_auc,mean_brier_score,mean_f1_50,mean_f1_calib,mean_calib_threshold,total_fit_time_sec
0,hist_gradient_boosting,0.610654,0.022871,0.584358,0.592605,0.854832,0.126631,0.562193,0.538861,0.534,190.357665
1,random_forest_candidate_b,0.582892,0.035178,0.529108,0.561034,0.836072,0.131976,0.537450,0.525200,0.508,790.045664
2,random_forest_candidate_a,0.581417,0.037692,0.524303,0.559061,0.834624,0.132852,0.533908,0.520586,0.504,682.002488
3,logistic_regression_standard,0.498995,0.048742,0.416555,0.479688,0.826452,0.138233,0.522714,0.515200,0.460,33.695080
4,logistic_regression_robust,0.498994,0.048713,0.416527,0.479676,0.826429,0.138187,0.522696,0.515659,0.458,199.291015


### 4. Market Capitalization Stratification

Evaluate HGBT performance across Large Cap, Mid Cap, and Small Cap segments.


In [5]:
preds_path = REPO_ROOT / 'data' / 'interim' / 'phase8_batch4' / 'supervised_predictions.parquet'
preds_df = pd.read_parquet(preds_path)

sub_df = preds_df.rename(columns={'y_true': 'hre_5d_train', 'prob_hist_gradient_boosting': 'hre_prob'})
cap_eval = ModelEvaluator.evaluate_by_cap_segment(
    sub_df, segment_col='cap_segment', target_col='hre_5d_train', prob_col='hre_prob', threshold=0.50
)
print('HGBT Market Cap Stratification:')
print(cap_eval.to_string(index=False))


HGBT Market Cap Stratification:
cap_segment   pr_auc  roc_auc  threshold       f1  precision   recall  balanced_accuracy  brier_score  log_loss    tp    fp     tn    fn  n_obs  n_pos  prevalence
      large 0.788097 0.938142        0.5 0.440680   0.922654 0.289469           0.642019     0.093816  0.291009  3018   253  46332  7408  57011  10426    0.182877
        mid 0.735477 0.919154        0.5 0.664140   0.557307 0.821646           0.836248     0.110930  0.354428 12268  9745  55592  2663  80268  14931    0.186014
      small 0.630354 0.884433        0.5 0.448538   0.292111 0.965655           0.737166     0.247167  0.690450  4358 10561  10934   155  26008   4513    0.173524
        ALL 0.592605 0.849174        0.5 0.560672   0.488620 0.657650           0.751777     0.126655  0.385807 19644 20559 112858 10226 163287  29870    0.182929


### 5. Phase 8 Batch 5 (Stage 8.12.4): Pre-Holdout Full-Population Training

In accordance with the Stage 8.12.3 configuration freeze and Stage 8.12.4 pre-holdout training protocol:
- All historical walk-forward validation experiments above are preserved and strictly **not rerun**.
- The Primary Champion architecture (`hist_gradient_boosting`, 36 features including F51 `relative_volume`) was fitted on the full eligible pre-holdout population (`2018-04-30` to `2025-10-10`, 466,346 rows, 1,841 sessions).
- Native integer histogram binning and missingness routing were applied without feature scaling or imputation.
- The boundary embargo (1,444 rows) and final holdout (74,334 rows) were strictly quarantined with zero access.

In [6]:
# Phase 8 Batch 5 (Stage 8.12.4): Inspect Pre-Holdout Champion HGBT Artifacts
import json
from pathlib import Path
import joblib
import pandas as pd

models_dir = REPO_ROOT / 'models' / 'phase8_batch5'
meta_dir = REPO_ROOT / 'reports' / 'phase8_batch5' / 'training_metadata'

m_name = 'hist_gradient_boosting'
meta_file = meta_dir / f'{m_name}_training_metadata.json'
with open(meta_file, 'r', encoding='utf-8') as f:
    meta = json.load(f)
art_path = models_dir / m_name / 'model.joblib'
pipeline = joblib.load(art_path)
hgbt_rec = [{
    'Model Identifier': meta['model_identifier'],
    'Role': 'PRIMARY SUPERVISED CHAMPION',
    'Algorithm': meta['algorithm'],
    'Feature Count': meta['feature_count'],
    'Training Start': meta['training_start_date'],
    'Training End': meta['training_end_date'],
    'Eligible Rows': f"{meta['eligible_training_rows']:,}",
    'Pos Class (HRE=1)': f"{meta['positive_class_count']:,} ({meta['positive_class_prevalence']:.2%})",
    'Fit Time (s)': f"{meta['fit_time_seconds']:.2f}s",
    'Artifact Path': str(art_path.relative_to(REPO_ROOT)),
    'Status': meta['training_completion_status'],
}]

print('=' * 80)
print('STAGE 8.12.4: PRE-HOLDOUT HISTGRADIENTBOOSTING CHAMPION SUMMARY')
print('=' * 80)
hgbt_df = pd.DataFrame(hgbt_rec)
print(hgbt_df.to_string(index=False))


STAGE 8.12.4: PRE-HOLDOUT HISTGRADIENTBOOSTING CHAMPION SUMMARY
      Model Identifier                        Role                      Algorithm  Feature Count Training Start Training End Eligible Rows Pos Class (HRE=1) Fit Time (s)                                            Artifact Path                 Status
hist_gradient_boosting PRIMARY SUPERVISED CHAMPION HistGradientBoostingClassifier             36     2018-04-30   2025-10-10       466,346   84,988 (18.22%)       10.20s models\phase8_batch5\hist_gradient_boosting\model.joblib COMPLETED_SUCCESSFULLY


### 6. Phase 8 Batch 5 (Stage 8.12.5): Supervised Holdout Evaluation — Primary Champion (HistGradientBoosting)

In Stage 8.12.5, the Primary Supervised Champion (`hist_gradient_boosting`, 36 features) was evaluated on the 250-trading-session holdout (`2025-10-20` to `2026-10-23`, 71,756 valid rows):

- Evaluates out-of-sample discrimination (PR-AUC, ROC-AUC) and calibration (Brier score, Log Loss).
- Evaluates market-cap segment performance across Large-Cap, Mid-Cap, and Small-Cap universes.
- Confirms whether HGBT maintains its decisive superiority over all competitor models on unseen data.

In [7]:
# Stage 8.12.5: Inspect Supervised Holdout Evaluation for Primary Champion HGBT
from pathlib import Path
import pandas as pd

metrics_path = REPO_ROOT / 'data' / 'interim' / 'phase8_batch5' / 'holdout' / 'supervised_holdout_metrics.csv'
holdout_metrics = pd.read_csv(metrics_path)
hgbt_holdout = holdout_metrics[holdout_metrics['model_name'] == 'hist_gradient_boosting'].copy()

display_cols = [
    'model_name', 'holdout_pr_auc', 'holdout_roc_auc', 'holdout_brier_score',
    'calibrated_threshold', 'f1_50', 'precision_50', 'recall_50',
    'f1_calib', 'precision_calib', 'recall_calib',
    'large_cap_pr_auc', 'mid_cap_pr_auc', 'small_cap_pr_auc'
]
print('=' * 80)
print('STAGE 8.12.5: HISTGRADIENTBOOSTING CHAMPION HOLDOUT PERFORMANCE')
print('=' * 80)
print(hgbt_holdout[display_cols].to_string(index=False))


STAGE 8.12.5: HISTGRADIENTBOOSTING CHAMPION HOLDOUT PERFORMANCE
            model_name  holdout_pr_auc  holdout_roc_auc  holdout_brier_score  calibrated_threshold    f1_50  precision_50  recall_50  f1_calib  precision_calib  recall_calib  large_cap_pr_auc  mid_cap_pr_auc  small_cap_pr_auc
hist_gradient_boosting        0.572332         0.845966             0.116693                 0.534 0.532121      0.486826   0.586709   0.53127         0.514227      0.549482          0.707882        0.689047          0.587886


### 7. Phase 8 Batch 5 (Stages 8.12.8 & 8.12.9): Final Model Selection & Champion Confirmation

#### 7.1 Authoritative Performance Summary
Across both expanding-window cross-validation and the untouched 250-session holdout, `hist_gradient_boosting` (Configuration `CFG_05_HGBT`, 36 features) achieved decisive out-of-sample supremacy:
* **Walk-Forward Mean PR-AUC:** **0.6107** (Pooled PR-AUC: 0.5926).
* **Holdout PR-AUC:** **0.5723** (Exceeds RF-B by +0.0343; exceeds Linear Baseline by +0.1672).
* **Holdout ROC-AUC:** **0.8460** (vs Walk-Forward mean 0.8548).
* **Holdout Brier Calibration Score:** **0.1167** (Lowest loss across all models; improved from 0.1266 in WF).
* **Holdout Log Loss:** **0.3625** (Lowest across all models; improved from 0.3857 in WF).
* **Monthly Subperiod Consistency:** Won **13 out of 13 monthly periods (100% win rate)** on the holdout partition.

#### 7.2 Walk-Forward vs. Holdout Generalization Assessment
| Metric | Walk-Forward 10-Fold Mean | Final Holdout (250 Sessions) | Absolute Delta | Retention % |
| :--- | :---: | :---: | :---: | :---: |
| **PR-AUC** | 0.6107 | 0.5723 | -0.0383 | **93.72%** |
| **ROC-AUC** | 0.8548 | 0.8460 | -0.0089 | **98.96%** |
| **Brier Score** | 0.1266 | 0.1167 | -0.0099 | Improved |
| **Log Loss** | 0.3857 | 0.3625 | -0.0232 | Improved |
| **F1 @ 0.50** | 0.5622 | 0.5321 | -0.0301 | 94.65% |
| **Calibrated F1 (tau*=0.534)** | 0.5389 | 0.5313 | -0.0076 | **98.59%** |

#### 7.3 Final Selection Decision and Architectural Rationale
1. **Decisive Leader:** Ranked #1 in both walk-forward cross-validation and final holdout evaluation (Kendall's $\tau = +1.000$).
2. **Minimal Generalization Decay:** Retained **93.72%** of its walk-forward PR-AUC, demonstrating the highest generalization fidelity of any model.
3. **Probability Calibration:** Achieved the lowest Brier score and Log Loss, guaranteeing accurate risk probability estimates for downstream position sizing.
4. **Institutional Large-Cap Performance:** Holdout PR-AUC of **0.7079** in the large-cap segment (+0.0870 over RF-B).
5. **Operational Suitability:** Full cross-sectional inference on 74,334 records completed in **0.22 seconds** (~3.0 $\mu$s per record), far exceeding the 1.0-second institutional latency standard.

**Final Decision:** `hist_gradient_boosting` (CFG_05_HGBT) is formally selected as the **Primary Champion Supervised Predictive Model** for forward liquidity risk intelligence. Formally recorded and frozen in `reports/phase8_batch5/final_model_selection_manifest.json`.